# CNN-Based Semiconductor Wafer Defect Detection — Colab Quickstart

**AI 570 — Team 4** (Paul, Pyle, Rajan, Rettura)

End-to-end training on the WM-811K dataset with a free Colab T4 GPU. ~15–25 minutes wall-clock for all three models at 20 epochs.

## Before you start

**Runtime → Change runtime type → Hardware accelerator: `T4 GPU`** (or L4 / A100 if Pro).

You need the `LSWMD_new.pkl` dataset (~2.2 GB). Either upload it directly or mount Google Drive if you've already stored it there.

## What this notebook does

1. Clone the repo
2. Install the package
3. Verify GPU
4. Mount / upload dataset
5. Run the full test suite
6. Train Custom CNN + ResNet-18 + EfficientNet-B0
7. Report metrics + save checkpoints to Drive

## 1. Clone and install

In [ ]:
REPO_URL = 'https://github.com/bpyle02/CNN-Based-Semiconductor-Wafer-Defect-Detection-Using-the-WM-811K-Dataset.git'
REPO_DIR = 'CNN-Based-Semiconductor-Wafer-Defect-Detection-Using-the-WM-811K-Dataset'

import os
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}
!git pull --ff-only
!pip install -q -e '.[dev]'

## 2. Verify GPU

In [ ]:
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('Mem:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('No GPU detected — switch runtime to T4 GPU before continuing.')

## 3. Load dataset

Pick **one** of the two cells below.

In [ ]:
# Option A: Mount Google Drive (assumes dataset at /MyDrive/datasets/LSWMD_new.pkl)
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
os.makedirs('data', exist_ok=True)
src = '/content/drive/MyDrive/datasets/LSWMD_new.pkl'
dst = 'data/LSWMD_new.pkl'
if not os.path.exists(dst):
    shutil.copy(src, dst)
print('Dataset size:', round(os.path.getsize(dst) / 1e9, 2), 'GB')

In [ ]:
# Option B: Upload from your computer (~2.2 GB — slow over typical connections)
# from google.colab import files
# import os, shutil
# os.makedirs('data', exist_ok=True)
# uploaded = files.upload()
# shutil.move(list(uploaded.keys())[0], 'data/LSWMD_new.pkl')
# print('Dataset uploaded to data/LSWMD_new.pkl')

In [ ]:
# Sanity-check the dataset
import pandas as pd
df = pd.read_pickle('data/LSWMD_new.pkl')
print('shape:', df.shape)
print('columns:', df.columns.tolist())
if 'failureType' in df.columns:
    print(df['failureType'].value_counts().head(10))

## 4. Run the test suite

In [ ]:
!pytest -q --maxfail=5

## 5. Train all three models on GPU

In [ ]:
EPOCHS = 20
BATCH_SIZE = 128
!python train.py --model all --epochs {EPOCHS} --batch-size {BATCH_SIZE} --device cuda

## 6. Inspect results

In [ ]:
import os, json, glob

for path in sorted(glob.glob('results/*_metrics.json')):
    with open(path) as f:
        m = json.load(f)
    print(f"{os.path.basename(path):40s} acc={m.get('accuracy',0):.4f} macro_f1={m.get('macro_f1',0):.4f}")

In [ ]:
from IPython.display import Image, display
import glob
for path in sorted(glob.glob('results/*_confusion_matrix.png') + glob.glob('results/*_training_curves.png')):
    print(path)
    display(Image(path))

## 7. Save checkpoints and results to Drive

In [ ]:
import os, shutil, time
out_dir = f"/content/drive/MyDrive/wafer_runs/{time.strftime('%Y%m%d_%H%M%S')}"
os.makedirs(out_dir, exist_ok=True)

for src_dir in ['checkpoints', 'results']:
    if os.path.isdir(src_dir):
        shutil.copytree(src_dir, os.path.join(out_dir, src_dir), dirs_exist_ok=True)
print('Saved to:', out_dir)

---

Done. Inspect the confusion matrices and per-class F1 scores — macro-F1 is the imbalance-aware metric; accuracy is dominated by the 85% `none` class.